In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
primaryPath.mkdir(parents=True, exist_ok=True)
evaluationRows = pl.read_parquet(dataPrepPath / 'evaluationRows.parquet')
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
featureSetAblation = predictions.join(evaluationRows.select('rowId', 'ndvi'), on='rowId', how='left').with_columns((pl.col('prediction') - pl.col('ndvi')).alias('error')).group_by('model', 'featureSet', 'seed').agg(pl.col('error').abs().mean().alias('mae'), pl.len().alias('rowCount')).sort('model', 'featureSet', 'seed')
featureSetAblation.write_parquet(primaryPath / 'featureSetAblation.parquet')
assert featureSetAblation.height == 2 * 4 * 5
assert featureSetAblation.select(pl.col('featureSet').n_unique()).item() == 4
featureSetAblation
